# k-Nearest Neighbors (kNN): Klassifikation des Iris-Datensatzes

**Grundlage:** *Introduction to Machine Learning with Python*, bereitgestellter Auszug, S. 20–26. Die Arbeitsaufträge knüpfen an das dort beschriebene Iris-Beispiel an.

Im Mittelpunkt steht die Vorhersage der Iris-Art aus vier Messwerten, die Aufteilung in Trainings- und Testdaten sowie die Evaluation mit Accuracy.

## 1. Daten laden und untersuchen

Der Iris-Datensatz enthält 150 Beobachtungen, vier numerische Merkmale und die drei Klassen `setosa`, `versicolor` und `virginica`. Die Daten werden direkt über `scikit-learn` geladen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

iris = load_iris()
X = iris.data
y = iris.target

print("Merkmale:", iris.feature_names)
print("Klassen:", iris.target_names)
print("Form von X:", X.shape)
print("Form von y:", y.shape)

In [ ]:
df = pd.DataFrame(X, columns=iris.feature_names)
df["species"] = iris.target_names[y]
df.head()

**Arbeitsauftrag:** Ordnen Sie die vier Messwerte den Merkmalen zu. Erläutern Sie den Unterschied zwischen `X` und `y` sowie die Bedeutung der Dimensionen `(150, 4)` und `(150,)`.

## 2. Visuelle Untersuchung

Die paarweisen Darstellungen entsprechen dem Grundgedanken der Abbildung auf S. 20 des Auszugs: Die Klassen lassen sich anhand der Messwerte unterschiedlich gut trennen.

In [ ]:
farben = np.array(["#4D7189", "#BA7059", "#7A8C69"])
abbildung = scatter_matrix(
    df[iris.feature_names],
    c=farben[y],
    figsize=(9, 9),
    diagonal="hist",
    alpha=0.75,
    s=18,
)
plt.suptitle("Iris: paarweise Zusammenhänge der Merkmale", y=1.01)
plt.show()

**Arbeitsauftrag:** Untersuchen Sie die Punktwolken. Welche beiden Merkmale scheinen die Arten besonders gut voneinander zu unterscheiden? Begründen Sie Ihre Einschätzung anhand der Darstellung.

## 3. Trainings- und Testdaten

Wie im Buch werden Trainings- und Testdaten mit `random_state=0` aufgeteilt. Ohne Angabe von `test_size` verwendet `train_test_split` standardmäßig 25 % der Daten für den Test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=0
)
print("Trainingsdaten:", X_train.shape, y_train.shape)
print("Testdaten:", X_test.shape, y_test.shape)

**Arbeitsauftrag:** Wie viele Iris-Blüten werden für den Aufbau des Modells verwendet? Wie viele Beobachtungen stehen für die anschließende Evaluation zur Verfügung?

## 4. kNN-Klassifikator mit k = 1

Für eine neue Beobachtung wird der nächste Trainingspunkt gesucht. Seine bekannte Iris-Art wird als Vorhersage übernommen.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X_train, y_train)

X_new = np.array([[5.0, 2.9, 1.0, 0.2]])
print("Form des neuen Datensatzes:", X_new.shape)
vorhersage = knn.predict(X_new)
print("Vorhergesagtes Klassenlabel:", vorhersage[0])
print("Vorhergesagte Iris-Art:", iris.target_names[vorhersage[0]])

**Arbeitsauftrag:** Erklären Sie, warum `X_new` die Form `(1, 4)` besitzt. Welche Bedeutung hat das Ergebnis `0`?

## 5. Distanzberechnung und Nachbarschaft

Die Suche nach dem nächsten Nachbarn lässt sich mit der euklidischen Distanz ausdrücken:

$$
d(\mathbf{x},\mathbf{z}) = \sqrt{\sum_{j=1}^{4}(x_j-z_j)^2}
$$

Mit:

- $d(\mathbf{x},\mathbf{z})$: Abstand zwischen zwei Iris-Beobachtungen;
- $x_j$ und $z_j$: Messwerte des $j$-ten Merkmals;
- $j=1,\ldots,4$: Index der vier numerischen Merkmale.

Die folgende Berechnung verwendet die tatsächlichen Iris-Trainingsbeobachtungen der Aufteilung aus Abschnitt 3.

In [ ]:
abstaende, nachbar_indices = knn.kneighbors(X_new, n_neighbors=3)
nachbarn = pd.DataFrame(
    X_train[nachbar_indices[0]],
    columns=iris.feature_names,
)
nachbarn["species"] = iris.target_names[y_train[nachbar_indices[0]]]
nachbarn["Distanz"] = abstaende[0]
nachbarn.index = ["A", "B", "C"]
nachbarn

In [ ]:
erster_nachbar = X_train[nachbar_indices[0, 0]]
differenzen = X_new[0] - erster_nachbar
quadrate = differenzen ** 2

print("Neue Blüte:", X_new[0])
print("Nächster Nachbar:", erster_nachbar)
print("Quadratische Differenzen:", quadrate)
print("Summe:", quadrate.sum())
print("Abstand:", np.sqrt(quadrate.sum()))

**Arbeitsauftrag:** Berechnen Sie die Abstände der neuen Blüte zu B und C auf Papier. Vergleichen Sie Ihre Ergebnisse mit der Tabelle. Welche Klasse würde sich bei `k=1` beziehungsweise `k=3` ergeben?

## 6. Evaluation des Klassifikators

Die Accuracy gibt den Anteil der richtig klassifizierten Testbeobachtungen an:

$$
\mathrm{Accuracy} = \frac{\text{Anzahl korrekt klassifizierter Beobachtungen}}{\text{Anzahl Testbeobachtungen}}
$$

In [ ]:
y_pred = knn.predict(X_test)
anzahl_richtig = np.sum(y_pred == y_test)
print("Richtig klassifizierte Blüten:", anzahl_richtig)
print("Anzahl Testbeobachtungen:", len(y_test))
print("Accuracy, manuell berechnet:", np.mean(y_pred == y_test))
print("Accuracy mit score():", knn.score(X_test, y_test))

**Arbeitsauftrag:** Erläutern Sie, weshalb die Accuracy anhand der Testdaten berechnet wird. Was bedeutet der Wert von ungefähr 0,97 für diese Teststichprobe?

## 7. Anzahl der Nachbarn verändern

Im Auszug wird erläutert, dass statt nur eines Nachbarn auch die drei oder fünf nächstgelegenen Trainingsbeobachtungen berücksichtigt werden können. Die häufigste Klasse bestimmt dann die Vorhersage.

In [ ]:
ergebnisse = []
for k in [1, 3, 5]:
    modell = KNeighborsClassifier(n_neighbors=k)
    modell.fit(X_train, y_train)
    ergebnisse.append({
        "k": k,
        "Vorhersage für neue Blüte": iris.target_names[modell.predict(X_new)[0]],
        "Accuracy Training": modell.score(X_train, y_train),
        "Accuracy Test": modell.score(X_test, y_test),
    })
pd.DataFrame(ergebnisse)

**Arbeitsauftrag:** Vergleichen Sie die Ergebnisse für `k=1`, `k=3` und `k=5`. Beschreiben Sie, welche Größe im Modell verändert wurde und welche Größen unverändert geblieben sind.

## 8. Entscheidungsflächen in zwei Merkmalen

Für eine zweidimensionale Darstellung werden ausschließlich Blütenblattlänge und Blütenblattbreite verwendet. Die Ergebnisse dieser Visualisierung sind deshalb **nicht** mit dem Klassifikator aus Abschnitt 4 gleichzusetzen, der alle vier Merkmale berücksichtigt.

In [ ]:
auswahl = [2, 3]
X_train_2d = X_train[:, auswahl]

x_grid, y_grid = np.meshgrid(
    np.linspace(0.5, 7.2, 150),
    np.linspace(-0.1, 2.9, 90),
)
gitter = np.c_[x_grid.ravel(), y_grid.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True, sharey=True)
for ax, k in zip(axes, [1, 5]):
    modell_2d = KNeighborsClassifier(n_neighbors=k)
    modell_2d.fit(X_train_2d, y_train)
    zuordnung = modell_2d.predict(gitter).reshape(x_grid.shape)
    ax.contourf(x_grid, y_grid, zuordnung,
                levels=[-0.5, 0.5, 1.5, 2.5],
                colors=["#4D7189", "#BA7059", "#7A8C69"], alpha=0.18)
    for klasse, bezeichnung in enumerate(iris.target_names):
        maske = y_train == klasse
        ax.scatter(X_train_2d[maske, 0], X_train_2d[maske, 1],
                   s=15, color=farben[klasse], label=bezeichnung)
    ax.set_title(f"k = {k}")
    ax.set_xlabel("Blütenblattlänge (cm)")
    ax.grid(alpha=0.12)
axes[0].set_ylabel("Blütenblattbreite (cm)")
axes[1].legend(frameon=False)
plt.tight_layout()
plt.show()

**Arbeitsauftrag:** Beschreiben Sie den Unterschied zwischen den Entscheidungsflächen bei `k=1` und `k=5`. Erläutern Sie, wie eine solche Fläche durch Klassifikationen vieler möglicher neuer Punkte entsteht.

## 9. Abschluss

Fassen Sie den Ablauf in eigenen Worten zusammen: von den Iris-Messwerten über `train_test_split`, `fit` und `predict` bis zur Evaluation mit `score`.